In [0]:
%pip install -q newspaper3k
%pip install -q lxml_html_clean
%pip install -q pymongo
%pip install -q "google-genai"
dbutils.library.restartPython()                                                                                             

In [0]:
from datetime import datetime
import requests, json
from newspaper import Article, Config
import pandas as pd
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, date_format, col, expr, to_date, lit

In [0]:
schema = StructType([
    StructField("date", StringType(), True),
    StructField("url", StringType(), True),
    StructField("domain", StringType(), True),
    StructField("outletName", StringType(), True),
    StructField("outletLogo", StringType(), True),
    StructField("outletTwitter", StringType(), True),
    StructField("title", StringType(), True),
    StructField("image", StringType(), True),
    StructField("desc", StringType(), True),
    StructField("lang", StringType(), True),
    StructField("author", StringType(), True),
])

In [0]:
unprocessed_files_df = spark.sql("""
    select file_name
    from news_app.default.gal_files
    where processed = false
    order by file_name
""").limit(1000)
# display(unprocessed_files_df)

file_paths = [f"/Volumes/news_app/default/news_app_volume/{row.file_name}" for row in unprocessed_files_df.collect()]

if file_paths:
    gal_df = spark.read.schema(schema).json(file_paths)
    # display(gal_df)
else:
    print("No unprocessed files found.")

In [0]:
outlet_names = [
    # "TV Guide", --Incomplete story
    # "quicknews-africa.net",
    # "The Nordic Page",
    "The Times of India",
    "Yahoo Finance",
    "DailyRidge.com - Fast-Factual-Free",
    # "BizToc", --Incomplete story
    "The Indian Express",
    "The Star",
    "The Economic Times",
    "Hindustan Times",
    "Mail Online",
    "Express.co.uk",
    "Free Press Journal",
    "Daily Mirror",
    "Moneycontrol",
    "The Hindu",
    "Yahoo News",
    "mint",
    "ABC News",
    "the Guardian",
    "Zee News",
    "The Mail",
    "bbc.com",
    "Fox News",
    "New York Post",
    "Middle East Star"
]

gal_df = gal_df.filter(gal_df['lang'] == 'en')
gal_df = gal_df.filter(gal_df.outletName.isin(outlet_names))
gal_df = gal_df.withColumn("added_timestamp", date_format(current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))
gal_df = gal_df.withColumn("partition_date", to_date("added_timestamp"))
gal_df = gal_df.withColumn("article_id", expr("uuid()"))
cols = ["article_id"] + [col for col in gal_df.columns if col != "article_id"]
gal_df = gal_df.select(*cols)

In [0]:
def parse_article(row):
    url = row['url']
    result = {
        'news_text': None,
        'image_url': None,
        # 'keywords': [],
        'published_timestamp': None
    }

    config = Config()
    config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'
    
    article = Article(url)
    article.download()
    article.parse()
    # article.nlp()
    result['news_text'] = article.text
    result['image_url'] = article.top_image
    # result['keywords'] = article.keywords if isinstance(article.keywords, list) else []
    result['published_timestamp'] = article.publish_date

    return result

pdf = gal_df.toPandas()

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

row_count = 0
success_count = 0
failure_count = 0
news_list = []

def process_row(row):
    row_dict = row.to_dict()
    try:
        result = {**row_dict, **parse_article(row_dict)}
        return (row_dict['url'], result, True)
    except Exception:
        return (row_dict['url'], None, False)

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {executor.submit(process_row, row): idx for idx, row in pdf.iterrows()}
    for future in as_completed(futures):
        url, result, success = future.result()
        row_count += 1
        print(row_count, url)
        if success:
            news_list.append(result)
            success_count += 1
        else:
            failure_count += 1

print(row_count, success_count, failure_count)

In [0]:
def get_llm_response(prompt):
    openrouter_api_key = dbutils.secrets.get(scope='news-app-secrets', key='openrouter-api-key')
    response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {openrouter_api_key}"
    },
    data=json.dumps({
        "model": "openai/gpt-3.5-turbo", # Optional
        "messages": [
        {
            "role": "user",
            "content": prompt
        }
        ]
    })
    )
    try:
        return response.json()['choices'][0]['message']['content']
    except:
        return None

In [0]:
def get_tags_prompt(title, desc, news_text):
    SYSTEM_INSTRUCTION = """
    Extract keywords from the document into a single, valid JSON object with top-level keys: 'people', 'location', and 'other'. Do not include markdown or text outside the JSON.
    LOCATIONS: Must be nested under keys: 'landmarks', 'localities', 'cities', 'states', and 'countries'. If a city is extracted, its state and country must also be provided (e.g., Chicago -> Illinois, USA).
    """

    FEW_SHOT_EXAMPLE_INPUT = """
    title: Student from Hyderabad ambushed and robbed at gunpoint in Chicago
    description: Syed Mazahir Ali, a resident of Hashim Nagar of Langer Houz, was a few minutes away from his flat in Campbell Ave, Chicago, when three armed robbers ambushed and attacked him
    content: A student from Hyderabad pursuing his Masters in Indiana Wesleyan University in Chicago was ambushed and robbed at gunpoint by three armed robbers on February 4. Syed Mazahir Ali, a resident of Hashim Nagar of Langer Houz, was a few minutes away from his flat in Campbell Ave, Chicago, when three armed robbers ambushed and attacked him. They fled with his wallet and mobile. There have been a string of violent incidents involving students from India over the past few weeks. In a viral video shot minutes after the attack, which is being circulated on social media, Mr. Ali can be heard mentioning how they jumped on him and that he is scared Speaking to The Hindu, his wife, Syeda Ruquiya Fatima Razvi, said that Mr. Ali went to the U.S. about six months ago for a two years Masters course in Information & Technology from Indiana Wesleyan University. “My husband sustained injuries on the back of his head, back and knees. He is admitted at a private hospital and is in a state of shock. His video doing rounds on social media is traumatising for us to watch,” she said. Ms. Rizwi has written to the Minister for External Affairs S. Jaishankar requesting help in getting the best medical treatment.
    """

    FEW_SHOT_EXAMPLE_OUTPUT = """
    {
        "people": [
            "Syed Mazahir Ali",
            "Syeda Ruquiya Fatima Razvi",
            "S. Jaishankar",
            "Minister for External Affairs",
            "Students from India"
        ],
        "location": {
            "landmarks": [],
            "localities": [
                "Hashim Nagar, Langer Houz",
                "Campbell Ave"
            ],
            "cities": [
                "Hyderabad",
                "Chicago"
            ],
            "states": [
                "Telangana",
                "Illinois"
            ],
            "countries": [
                "India",
                "USA"
            ]
        },
        "other": [
            "robbery",
            "gunpoint",
            "attack",
            "violence",
            "injuries",
            "hospital",
            "video",
            "social media",
            "medical treatment",
            "Indiana Wesleyan University",
            "Masters course in Information & Technology"
        ]
    }
    """

    tags_prompt = f"""
    {SYSTEM_INSTRUCTION}

    --- START EXAMPLE ---
    INPUT:
    {FEW_SHOT_EXAMPLE_INPUT}

    OUTPUT JSON:
    {FEW_SHOT_EXAMPLE_OUTPUT}
    --- END EXAMPLE ---

    --- NEW INPUT ---
    INPUT DOCUMENT:
    title: {title}
    description: {desc}
    content: {news_text}

    OUTPUT JSON:
    """
    
    return tags_prompt

In [0]:
def get_category_prompt(title, desc):
    SYSTEM_INSTRUCTION = """
    You are an expert news classifier. Your SOLE task is to analyze the provided news title and description
    and output the single, best-fitting category from the list.
    Your response MUST be one word and MUST be from the list of provided Categories.
    """

    FEW_SHOT_EXAMPLE_INPUT = """
    Title: Student from Hyderabad ambushed and robbed at gunpoint in Chicago
    Description: Syed Mazahir Ali, a resident of Hashim Nagar of Langer Houz, was a few minutes away from his flat in Campbell Ave, Chicago, when three armed robbers ambushed and attacked him
    """

    FEW_SHOT_EXAMPLE_OUTPUT = "Crime"

    category_prompt = f"""
    {SYSTEM_INSTRUCTION}

    --- AVAILABLE CATEGORIES ---
    Technology, Environment, Entertainment, Politics, Education, Crime, Sports, Business, Travel, Money and Nation.

    --- CLASSIFICATION RULE ---
    Anything that involves government bodies and similar news belongs to Nation.

    --- START EXAMPLE ---
    INPUT:
    {FEW_SHOT_EXAMPLE_INPUT}

    OUTPUT:
    {FEW_SHOT_EXAMPLE_OUTPUT}
    --- END EXAMPLE ---

    --- NEW INPUT ---
    INPUT:
    Title: {title}
    Description: {desc}

    OUTPUT:
    """

    return category_prompt

In [0]:
def get_tags(title, desc, news_text):

    tags = {}

    tags_prompt = get_tags_prompt(title, desc, news_text)
    tags = get_llm_response(tags_prompt)

    if tags:
        # Check for markdown code fence and strip it
        if tags.strip().startswith("```"):
            start = tags.find('{')
            end = tags.rfind('}') + 1
            if start != -1 and end != -1:
                tags = tags[start:end]

        if tags and tags[0] == '{':
            tags = json.loads(tags)

    return tags
    
def get_category(title, desc):
    category = ''

    category_prompt = get_category_prompt(title, desc)
    category = get_llm_response(category_prompt)

    if category:
        category = category.strip()
    allowed_categories = {"Technology", "Environment", "Entertainment", "Politics", "Education", "Crime", "Sports", "Business", "Travel", "Money", "Nation"}
    if category in allowed_categories:
        category = category
        
    return category

In [0]:
news_count = 0
success_count = 0
failure_count = 0
for i in range(len(news_list)):
    tags = get_tags(news_list[i]['title'], news_list[i]['desc'], news_list[i]['news_text'])
    category = get_category(news_list[i]['title'], news_list[i]['desc'])
    news_count += 1
    if tags == {} or category == '':
        failure_count += 1
    else:
        success_count += 1
    news_list[i]['tags'] = tags
    news_list[i]['category'] = category
    news_list[i]['added_timestamp'] = str(news_list[i]['added_timestamp'])
    news_list[i]['partition_date'] = str(news_list[i]['partition_date'])
    news_list[i]['published_timestamp'] = news_list[i]['published_timestamp'].isoformat().replace("+05:30", "Z") if news_list[i]['published_timestamp'] is not None and isinstance(news_list[i]['published_timestamp'], datetime) else news_list[i]['published_timestamp']
    print("Current News Count:", news_count)
print("Total News:", news_count)
print("Success:", success_count)
print("Failure:", failure_count)

In [0]:
from pymongo import MongoClient

mongo_uri = dbutils.secrets.get(scope="news-app-secrets", key="mongo-uri")
client = MongoClient(mongo_uri)

db = client["news_app"]
collection = db["english_news"]

collection.insert_many(news_list)

# remove _id field that was inserted into news_list when inserting into MongoDB
for i in range(len(news_list)):
    news_list[i].pop('_id', None)

In [0]:
news_pdf = pd.DataFrame(news_list)

# Create a DataFrame from the nested 'tags' column
tags_normalized = pd.json_normalize(news_pdf['tags'], sep='_')

# The original 'tags' column is now redundant
news_pdf = news_pdf.drop('tags', axis=1)

# Join the normalized columns back to the main DataFrame
news_pdf = news_pdf.join(tags_normalized)

gal_processed_df = spark.createDataFrame(news_pdf)

In [0]:
gal_processed_df.write.mode("append").partitionBy("partition_date").saveAsTable("news_app.default.gdelt_gal")

In [0]:
unprocessed_files_df = unprocessed_files_df.withColumn('processed', lit(True))
unprocessed_files_df.createOrReplaceTempView("unprocessed_files_update")

spark.sql("""
MERGE INTO news_app.default.gal_files AS target
USING unprocessed_files_update AS source
ON target.file_name = source.file_name
WHEN MATCHED THEN
  UPDATE SET target.processed = source.processed
""")